In [3]:
import monpa

text = "老實說，這部電影真的非常難看又無聊，但是那裡的餐點還算可以。"

# monpa.pseg 會直接回傳 (詞, 詞性) 的列表
result = monpa.pseg(text)

print("【每個詞對應的詞性標註】:")
print("-" * 30)
for word, flag in result:
    print(f"{word:<6} --> 詞性標記: {flag}")

# 保留名詞(N)、形容詞(A)、動詞(V)
# 註：Monpa 的標籤格式為 VA(狀態動詞/形容詞)、Na(普通名詞)、VK(狀態動詞) 等
target_pos = ('N', 'A', 'V')
filtered = [w for w, f in result if f.startswith(target_pos)]

print("\n【過濾後的結果】:", filtered)

+---------------------------------------------------------------------+
  Welcome to MONPA: Multi-Objective NER POS Annotator for Chinese
+---------------------------------------------------------------------+
已找到 model檔。Found model file.
【每個詞對應的詞性標註】:
------------------------------
老實說    --> 詞性標記: Dk
，      --> 詞性標記: COMMACATEGORY
這      --> 詞性標記: Nep
部      --> 詞性標記: Nf
電影     --> 詞性標記: Na
真的     --> 詞性標記: D
非常     --> 詞性標記: Dfa
難看     --> 詞性標記: VH
又      --> 詞性標記: D
無聊     --> 詞性標記: VH
，      --> 詞性標記: COMMACATEGORY
但是     --> 詞性標記: Cbb
那裡     --> 詞性標記: Ncd
的      --> 詞性標記: DE
餐點     --> 詞性標記: Na
還      --> 詞性標記: D
算      --> 詞性標記: VG
可以     --> 詞性標記: VH
。      --> 詞性標記: PERIODCATEGORY

【過濾後的結果】: ['這', '部', '電影', '難看', '無聊', '那裡', '餐點', '算', '可以']


In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer


class PurePromptTester:
    def __init__(self, model_name_or_path: str):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name_or_path, 
            trust_remote_code=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None,
            trust_remote_code=True
        )
        if self.device == "cpu":
            self.model.to(self.device)
        self.model.eval()

    def get_metrics(self, text: str):
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        input_ids = inputs["input_ids"]

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits

        # 計算資訊熵 (Entropy)
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)
        token_entropies = -torch.sum(probs * log_probs, dim=-1).squeeze(0)

        # 計算困惑度 (Perplexity)
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)), 
            shift_labels.view(-1)
        )
        ppl = torch.exp(loss.mean()).item()

        # 取得最高熵的位置與 Token
        max_entropy_idx = torch.argmax(token_entropies).item()
        max_entropy_token_id = input_ids[0, max_entropy_idx].item()
        max_entropy_token_str = self.tokenizer.decode([max_entropy_token_id])

        return {
            "ppl": round(ppl, 4),
            "max_entropy_val": round(token_entropies[max_entropy_idx].item(), 4),
            "max_entropy_token": max_entropy_token_str,
            "max_entropy_index": max_entropy_idx
        }


if __name__ == "__main__":
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # 可替換為任何本地或線上模型
    
    # 讓使用者自行輸入 2 個 Prompt
    prompt1 = input("Enter Prompt 1: ")
    prompt2 = input("Enter Prompt 2: ")

    tester = PurePromptTester(model_name_or_path=MODEL_NAME)

    res1 = tester.get_metrics(prompt1)
    res2 = tester.get_metrics(prompt2)

    # 僅輸出數據結果，無解說詞
    print("\n--- RESULTS ---")
    print(f"P1_PPL: {res1['ppl']} | P1_MaxEntropy: {res1['max_entropy_val']} | P1_MaxToken: '{res1['max_entropy_token']}' (Index: {res1['max_entropy_index']})")
    print(f"P2_PPL: {res2['ppl']} | P2_MaxEntropy: {res2['max_entropy_val']} | P2_MaxToken: '{res2['max_entropy_token']}' (Index: {res2['max_entropy_index']})")

d:\2.programm2\github-star\presidio-research\venvpr\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\Tools\huggingface_cache\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to 


--- RESULTS ---
P1_PPL: 37.4968 | P1_MaxEntropy: 6.9023 | P1_MaxToken: '的' (Index: 18)
P2_PPL: 41.2504 | P2_MaxEntropy: 6.3196 | P2_MaxToken: '，' (Index: 10)
